## Create Stimuli List

Create the stimuli lists from the final cue-target pairs and the words that have not been normed. This file assumes a 200 word rating set and will randomly split them into chunks. 

### Libraries and Functions

In [ ]:
import os
import random
import pandas as pd


def build_norming_lists_from_updates(
    pair_updates_path,
    variables_normed_path,
    lang,
    output_base="02-Stimuli",
    bucket_size=200,
    seed=42,
):
    rng = random.Random(seed)
    tasks = ["aoa", "image", "concrete", "valence", "arousal", "familiar"]

    # -------------------------
    # 1) Load data
    # -------------------------
    pairs = pd.read_csv(pair_updates_path)
    normed = pd.read_csv(variables_normed_path)

    # -------------------------
    # 2) Required columns
    # -------------------------
    target_col = f"{lang}_target"
    final_cue_col = f"final_{lang}_cue"

    if target_col not in pairs.columns:
        raise ValueError(f"Missing column: {target_col}")

    if final_cue_col not in pairs.columns:
        raise ValueError(f"Missing column: {final_cue_col}")

    # -------------------------
    # 3) Clean helper
    # -------------------------
    def clean(s):
        return (
            s.dropna()
            .astype(str)
            .str.strip()
            .replace("", pd.NA)
            .dropna()
        )

    # -------------------------
    # 4) Candidate pool
    # -------------------------
    candidate_words = pd.concat(
        [
            clean(pairs[target_col]),
            clean(pairs[final_cue_col]),
        ],
        ignore_index=True,
    )

    candidate_words = candidate_words.drop_duplicates().tolist()
    candidate_set = set(candidate_words)

    # -------------------------
    # 5) Normalize normed sheet
    # -------------------------
    if "word" not in normed.columns:
        raise ValueError("variables_normed must contain 'word' column")

    normed["word"] = clean(normed["word"])

    for task in tasks:
        normed[task] = normed[task].astype(str).str.lower().map(
            {"true": True, "false": False, "1": True, "0": False}
        )

    normed = normed.drop_duplicates(subset="word")

    # -------------------------
    # 6) Build task lists (False = needed)
    # -------------------------
    task_word_lists = {}

    for task in tasks:
        needed = normed.loc[normed[task] == False, "word"].tolist()

        # only keep words present in stimuli
        needed = [w for w in needed if w in candidate_set]

        # dedupe while preserving order
        needed = list(dict.fromkeys(needed))

        task_word_lists[task] = needed

    # -------------------------
    # 7) Bucket function
    # -------------------------
    def make_buckets(words):
        words = list(words)
        rng.shuffle(words)

        buckets = []
        used = set()

        for i in range(0, len(words), bucket_size):
            chunk = words[i:i + bucket_size]
            used.update(chunk)

            # pad if needed
            if len(chunk) < bucket_size:
                needed_n = bucket_size - len(chunk)

                fillers = [w for w in candidate_words if w not in used]
                rng.shuffle(fillers)

                chunk.extend(fillers[:needed_n])
                used.update(chunk)

            buckets.append(chunk)

        return buckets

    task_buckets = {
        task: make_buckets(task_word_lists[task])
        for task in tasks
    }

    # -------------------------
    # 8) Save
    # -------------------------
    for task, buckets in task_buckets.items():
        task_dir = os.path.join(output_base, lang, task)
        os.makedirs(task_dir, exist_ok=True)

        for i, bucket in enumerate(buckets, start=1):
            pd.DataFrame({"word": bucket}).to_csv(
                os.path.join(task_dir, f"{task}_list_{i}.csv"),
                index=False
            )

    # -------------------------
    # 9) Debug output
    # -------------------------
    print(f"Seed: {seed}")
    print(f"Candidate pool: {len(candidate_words)}")

    for task in tasks:
        sizes = [len(b) for b in task_buckets[task]]
        print(f"{task}: needed={len(task_word_lists[task])}, buckets={sizes}")

    return task_buckets

### Build Stimuli Lists

In [ ]:

task_buckets = build_norming_lists_from_updates(
    pair_updates_path="ukr/ukr_word_pairs_update.csv",
    variables_normed_path="ukr/ukr_variables_normed.csv",
    lang="ukr",
    output_base="../02-Stimuli",
    bucket_size=200,
    seed=4536
)



Seed: 4536
Candidate pool: 1939
aoa: needed=1939, buckets=[400, 400, 400, 400, 339]
image: needed=1939, buckets=[400, 400, 400, 400, 339]
concrete: needed=1939, buckets=[400, 400, 400, 400, 339]
valence: needed=1939, buckets=[400, 400, 400, 400, 339]
arousal: needed=1939, buckets=[400, 400, 400, 400, 339]
familiar: needed=1939, buckets=[400, 400, 400, 400, 339]
